In [74]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)

In [75]:
df = pd.read_csv('loan_data.csv')

In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [77]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [78]:
X = df.drop(columns="loan_status")
y = df['loan_status']

In [79]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [80]:
xtrain,xtest,ytrain,ytest = train_test_split(
    X,y,train_size=0.8,random_state=42
)

In [81]:
cat_cols = X.select_dtypes(include='string').columns
cat_cols

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='str')

In [82]:
num_cols = X.select_dtypes(include='number').columns
num_cols

Index(['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score'],
      dtype='str')

In [83]:
X[cat_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [84]:
xtrain[cat_cols]

,person_gender,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
25180,female,Bachelor,MORTGAGE,PERSONAL,No
12555,male,High School,RENT,VENTURE,Yes
29153,female,Master,MORTGAGE,PERSONAL,Yes
23838,male,Associate,RENT,EDUCATION,No
35686,male,Master,RENT,HOMEIMPROVEMENT,Yes
...,...,...,...,...,...
11284,male,High School,MORTGAGE,EDUCATION,Yes
44732,female,High School,MORTGAGE,HOMEIMPROVEMENT,No
38158,female,Associate,MORTGAGE,VENTURE,Yes
860,male,Master,RENT,DEBTCONSOLIDATION,No


In [85]:
cat_cols

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='str')

In [86]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [87]:
ordinal_col = ['person_education']

ordinal_encoder = OrdinalEncoder(
    categories=[['High School', 'Bachelor', 'Master', 'Associate', 'Doctorate']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)


xtrain_oe_values = ordinal_encoder.fit_transform(xtrain[ordinal_col])
xtest_oe_values = ordinal_encoder.transform(xtest[ordinal_col]) 


xtrain_oe_df = pd.DataFrame(data=xtrain_oe_values, columns=ordinal_col).reset_index(drop=True)
xtest_oe_df = pd.DataFrame(data=xtest_oe_values, columns=ordinal_col).reset_index(drop=True)


In [88]:

ohe_cols = [col for col in cat_cols if col != 'person_education']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='error')

xtrain_ohe_values = ohe.fit_transform(xtrain[ohe_cols])
xtest_ohe_values = ohe.transform(xtest[ohe_cols])

xtrain_ohe_df = pd.DataFrame(
    data=xtrain_ohe_values, 
    columns=ohe.get_feature_names_out()
).reset_index(drop=True)

xtest_ohe_df = pd.DataFrame(
    data=xtest_ohe_values, 
    columns=ohe.get_feature_names_out()
).reset_index(drop=True)


In [89]:
xtrain = pd.concat([
    xtrain[num_cols].reset_index(drop=True),
    xtrain_oe_df,
    xtrain_ohe_df
], axis=1)

xtest = pd.concat([
    xtest[num_cols].reset_index(drop=True),
    xtest_oe_df,
    xtest_ohe_df
], axis=1)


In [90]:
xtrain

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,person_education,person_gender_female,...,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,34.0,97265.0,11,15000.0,12.73,0.15,9.0,631,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,25.0,72953.0,3,12000.0,11.86,0.16,4.0,659,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,41.0,322597.0,18,24000.0,10.37,0.07,11.0,683,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,27.0,94232.0,4,9600.0,17.14,0.10,7.0,641,3.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,27.0,84873.0,7,7059.0,12.97,0.08,3.0,706,2.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35995,26.0,88451.0,6,10625.0,6.03,0.12,4.0,559,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
35996,25.0,34772.0,3,5876.0,11.69,0.17,4.0,647,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
35997,33.0,58317.0,9,10707.0,10.74,0.18,9.0,652,3.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
35998,26.0,178602.0,6,20000.0,17.99,0.11,3.0,604,2.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [91]:
xtest

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,person_education,person_gender_female,...,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,32.0,96865.0,10,7500.0,6.04,0.08,10.0,601,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,24.0,56838.0,6,9000.0,11.49,0.16,4.0,647,3.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,22.0,37298.0,0,5000.0,14.88,0.13,4.0,711,2.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,23.0,39944.0,1,5000.0,13.99,0.13,3.0,597,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,42.0,67974.0,20,10000.0,15.41,0.15,11.0,586,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,21.0,133205.0,0,14000.0,9.99,0.11,2.0,654,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
8996,39.0,61889.0,17,6725.0,16.45,0.11,11.0,606,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
8997,24.0,78950.0,1,14000.0,11.65,0.18,3.0,614,2.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
8998,38.0,37142.0,16,8000.0,11.01,0.22,14.0,695,3.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [92]:
model = LogisticRegression(max_iter=5000)
model.fit(xtrain,ytrain)

c:\Users\91815\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [93]:
ytrain_pred = model.predict(xtrain)
ytest_pred=model.predict(xtest)

CONFUSION MATRIX

In [94]:
# FOR TRAIN DATA

c_m_train= confusion_matrix(ytrain,ytrain_pred)
c_m_train

array([[26024,  1986],
       [ 2047,  5943]])

In [95]:
# FOR TEST DATA

c_m_test= confusion_matrix(ytest,ytest_pred)
c_m_test

array([[6482,  508],
       [ 521, 1489]])

In [96]:
y.unique()

array([1, 0])

PRECISION

In [97]:
# FOR TRAIN DATA

p_s_train_pos = precision_score(ytrain, ytrain_pred, pos_label=1)
p_s_train_pos

0.7495270525917518

In [98]:
# FOR TRAIN DATA

p_s_train_neg = precision_score(ytrain,ytrain_pred,pos_label=0)
p_s_train_neg

0.9270777670905918

In [99]:
# FOR TEST DATA

p_s_test_pos = precision_score(ytest,ytest_pred,pos_label=1)
p_s_test_pos

0.7456184276414622

In [100]:
# FOR TEST DATA

p_s_test_neg = precision_score(ytest,ytest_pred,pos_label=0)
p_s_test_neg

0.9256033128659146

RECALL SCORE

In [101]:
# FOR TRAIN DATA

p_s_train_pos= recall_score(ytrain,ytrain_pred,pos_label=1)
p_s_train_pos

0.7438047559449311

In [102]:
# FOR TRAIN DATA

p_s_train_neg = recall_score(ytrain,ytrain_pred,pos_label=0)
p_s_train_neg

0.9290967511602999

In [103]:
# FOR TEST DATA

p_s_test_pos = recall_score(ytest,ytest_pred,pos_label=1)
p_s_test_pos

0.7407960199004975

In [104]:
# FOR TEST DATA

p_s_test_neg= recall_score(ytest,ytest_pred,pos_label=0)
p_s_test_neg

0.9273247496423462

F1_SCORE

In [105]:
y.value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [106]:
f1_score(ytest,ytest_pred,pos_label=0)

0.9264632316158079

In [107]:
f1_score(ytest,ytest_pred,pos_label=1)

0.7431994010481657

CLASSIFICATION REPORT

In [108]:
from sklearn.metrics import classification_report
report=classification_report(ytrain,ytrain_pred)
print(report)

              precision    recall  f1-score   support

           0       0.93      0.93      0.93     28010
           1       0.75      0.74      0.75      7990

    accuracy                           0.89     36000
   macro avg       0.84      0.84      0.84     36000
weighted avg       0.89      0.89      0.89     36000



In [109]:
report_test=classification_report(ytest,ytest_pred)
print(report_test)

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      6990
           1       0.75      0.74      0.74      2010

    accuracy                           0.89      9000
   macro avg       0.84      0.83      0.83      9000
weighted avg       0.89      0.89      0.89      9000



CONCLUSION

**TRAIN DATA**

*RECALL SCORE*

* OUT OF ALL ACTUAL'0' SAMPLES,95% ARE PREDICTED CORRECTLY AS '0',REMAINING ARE WRONGLY PREDICTED AS '1'
* OUT OF ALL ACTUAL '1' SAMPLES,42% ARE CORRECTLY PREDICTED AS 1,REMAINING ARE WRONGLY PREDICTED AS '0'

*PRECISION SCORE*

* OUT OF ALL PREDICTED '0' SAMPLES,85% ARE CORRECTLY PREDICTED AS '0',REMAINING ARE WRONGLY PREDICTED AS '0'
* * OUT OF ALL PREDICTED '1' SAMPLES,72% ARE CORRECTLY PREDICTED AS '1',REMAINING ARE WRONGLY PREDICTED AS '1'

*F1 SCORE*

* MODEL IS BALANCING RECALL AND PRECISION FOR '0' SAMPLES
* MODEL IS NOT BALANCING RECALL AND PRECISION FOR '1' SAMPLE

Weighted Average=(precision of label o)